# Подготовка модели распознавания рукописных букв и цифр

In [16]:
# Импорты для анализа данных
import torch
import torchvision

# Загрузка данных
dataset = torchvision.datasets.EMNIST('data/', split='balanced', download=False)

# Анализ данных
print(f"Количество образцов: {len(dataset)}")
print(f"Размер изображения: {dataset[0][0].size}")
print(f"Количество классов: {len(dataset.classes)}")


Количество образцов: 112800
Размер изображения: (28, 28)
Количество классов: 47


In [17]:
# Импорты для предобработки данных
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms

# Преобразования с аугментацией
transform_train = transforms.Compose([
    transforms.RandomRotation(10),  # случайные повороты до 10 градусов
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # случайные сдвиги
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # нормализация для EMNIST
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # та же нормализация для теста
])

# Загрузка с преобразованиями
train_dataset = torchvision.datasets.EMNIST('data/', download=False, 
                                           split='balanced', transform=transform_train)
test_dataset = torchvision.datasets.EMNIST('data/', download=False, 
                                          split='balanced', transform=transform_test, train=False)

# Разделение на выборки
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_subset, val_dataset = random_split(train_dataset, [train_size, val_size])

# DataLoader с увеличенным batch_size
batch_size = 128  # увеличено с 64 до 128
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [18]:
# Импорты для построения модели
import torch.nn as nn
import torch.nn.functional as F

class CNNModel(nn.Module):
    def __init__(self, num_classes=47):
        super(CNNModel, self).__init__()
        
        # Сверточные слои
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Пулинг
        self.pool = nn.MaxPool2d(2, 2)
        
        # Полносвязные слои
        self.fc1 = nn.Linear(128 * 3 * 3, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)
        
    def forward(self, x):
        # Сверточные слои с активацией и пулингом
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Преобразование в вектор
        x = x.view(-1, 128 * 3 * 3)
        
        # Полносвязные слои
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Создание модели
model = CNNModel(num_classes=47)
print(model)

CNNModel(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1152, out_features=512, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=47, bias=True)
)


In [20]:
# Импорты для обучения модели
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Параметры обучения
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # добавлен weight_decay

# Learning rate scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

num_epochs = 100
best_val_acc = 0.0
patience = 10
patience_counter = 0

train_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    # Обучение
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # Валидация
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    
    train_losses.append(avg_loss)
    val_accuracies.append(val_accuracy)
    
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')
    
    # Learning rate scheduler
    scheduler.step(val_accuracy)
    
    # Сохранение лучшей модели
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model.state_dict(), 'model.ckpt')
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

print(f'Best validation accuracy: {best_val_acc:.2f}%')

Epoch 1/100, Loss: 0.9843, Val Accuracy: 81.99%
Epoch 2/100, Loss: 0.5530, Val Accuracy: 84.77%
Epoch 3/100, Loss: 0.4938, Val Accuracy: 86.22%
Epoch 4/100, Loss: 0.4574, Val Accuracy: 87.12%
Epoch 5/100, Loss: 0.4321, Val Accuracy: 87.22%
Epoch 6/100, Loss: 0.4153, Val Accuracy: 87.06%
Epoch 7/100, Loss: 0.4039, Val Accuracy: 87.96%
Epoch 8/100, Loss: 0.3929, Val Accuracy: 87.48%
Epoch 9/100, Loss: 0.3845, Val Accuracy: 87.71%
Epoch 10/100, Loss: 0.3788, Val Accuracy: 88.06%
Epoch 11/100, Loss: 0.3698, Val Accuracy: 87.58%
Epoch 12/100, Loss: 0.3665, Val Accuracy: 87.31%
Epoch 13/100, Loss: 0.3614, Val Accuracy: 88.39%
Epoch 14/100, Loss: 0.3591, Val Accuracy: 87.59%
Epoch 15/100, Loss: 0.3508, Val Accuracy: 88.14%
Epoch 16/100, Loss: 0.3513, Val Accuracy: 87.69%
Epoch 17/100, Loss: 0.3436, Val Accuracy: 88.44%
Epoch 18/100, Loss: 0.3413, Val Accuracy: 88.51%
Epoch 19/100, Loss: 0.3419, Val Accuracy: 88.50%
Epoch 20/100, Loss: 0.3402, Val Accuracy: 88.52%
Epoch 21/100, Loss: 0.3365, V

In [21]:
# Импорты для тестирования модели
from sklearn.metrics import classification_report
import numpy as np

# Загрузка лучшей модели
model.load_state_dict(torch.load('model.ckpt'))
model.eval()

# Тестирование
all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Метрики
accuracy = sum(np.array(all_predictions) == np.array(all_labels)) / len(all_labels)
print(f'Test Accuracy: {accuracy:.4f}')

# Сохранение classification report в файл (встроенная функция sklearn)
with open('classification_report.txt', 'w') as f:
    f.write(f'Test Accuracy: {accuracy:.4f}\n\n')
    f.write(classification_report(all_labels, all_predictions))

Test Accuracy: 0.9043
